# Affinity graph diagnostic — does spatial-context smoothing wash out cell-type structure?

Follow-up to `plan1_niche_recovery_eval.ipynb`, which found scProto's metacells have
*higher* niche purity than SEACells (0.78 vs 0.51 average) but *far worse* gene-level niche
recovery — traced to scProto's metacells being extremely size-skewed (median 1-4 real member
cells in most (cell type, niche) groups, vs. SEACells' 26-190), consistent with a chunk of
scProto's 300 prototypes being under-used (`n_unused_protos`/`unused_proto_ratio`, already
tracked elsewhere in this codebase).

**Hypothesis being tested here**: the affinity graph used for the evaluated scProto run
(`affinity_type="ctx"`, `interpretable_ssl/augmenters/graph_generator.py:448`) builds its
kernel on `X_ctx` — an **unweighted mean of `X_pca` over every spatial neighbor within a
7.5-unit radius, with no cell-type filtering** (`build_context`/`spatial_context_pca`,
`graph_generator.py:657-706`). Two cells that are spatially close share most of the same
neighbor set, so averaging pulls their `X_ctx` vectors toward each other regardless of cell
type — this should show up as: (a) a denser / more uniform graph than the plain-`X_pca`
`arbf` graph, and (b) same-cell-type vs. cross-cell-type edge weights that are much harder to
tell apart in the `ctx` graph than in `arbf`.

Both graphs are already computed and saved — this notebook only loads and analyzes them, no
retraining, no affinity regeneration.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Same numpy-safe install sequence as plan1_niche_recovery_eval.ipynb / the other
# notebooks in this folder -- a plain "pip install ... SEACells" alone leaves numpy in a
# state that breaks anndata's import ("cannot import name '_center' from numpy._core.umath")
# until pinned back down. This notebook only needs numpy/scipy/pandas/matplotlib to unpickle
# and analyze the saved sparse affinity matrices, but nb_setup.py (run below, for DATASETS /
# CODE_DIR / adata loading convenience) unconditionally imports the full eval stack.
!pip install -q scarches faiss-cpu scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

In [ ]:
# IMPORTANT: restart the runtime now (Runtime -> Restart session) before running the
# cells below -- numpy/scipy/anndata are C-extension linked, so an in-process upgrade
# alone will not reliably take effect on already-imported modules.

In [ ]:
_checks = {
    "numpy": "numpy", "scipy": "scipy", "anndata": "anndata", "scanpy": "scanpy",
    "scarches": "scarches", "scvi-tools": "scvi", "seacells": "SEACells",
    "palantir": "palantir", "scib-metrics": "scib_metrics", "faiss": "faiss",
}
_failed = []
for pkg, mod in _checks.items():
    try:
        __import__(mod)
    except ImportError as e:
        _failed.append((pkg, str(e)))
if _failed:
    print("FAILED imports (fix before continuing, then restart runtime again):")
    for pkg, err in _failed:
        print(f"  {pkg}: {err}")
else:
    print("All packages import cleanly.")

In [ ]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

In [ ]:
import os
import pickle as pkl

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import scipy.sparse as sp

from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import CODE_DIR

DS_ID = 'ss28nsc'
CT_KEY = 'celltypes'

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.dpi'] = 150

## Load dataset (cell types only needed) + both saved affinity graphs

In [ ]:
adata = sc.read_h5ad(DATASETS[DS_ID]['path'])
celltypes = adata.obs[CT_KEY].to_numpy()
print(adata.shape, 'cells x genes;', len(np.unique(celltypes)), 'cell types')

GRAPH_DIR = os.path.join(CODE_DIR, 'graphs')
GRAPHS = {
    'arbf (raw X_pca, no spatial pooling)':
        os.path.join(GRAPH_DIR, 'affinity_ss28nsc28804_ncomp50_kneighbors50_arbf.pkl'),
    'ctx (mean X_pca over spatial neighbors -- the evaluated scProto run)':
        os.path.join(GRAPH_DIR, 'affinity_ss28nsc28804_ncomp50_kneighbors50_ctx.pkl'),
}

aff = {}
for name, path in GRAPHS.items():
    with open(path, 'rb') as f:
        M = pkl.load(f)
    aff[name] = sp.csr_matrix(M)
    print(f'{name}: shape={aff[name].shape}  nnz={aff[name].nnz}  '
          f'density={aff[name].nnz / np.prod(aff[name].shape):.4%}')

## Degree (row nnz) and edge-weight distributions

If spatial-context smoothing flattens the graph, `ctx` should show a higher/denser degree
distribution and/or a less spread-out edge-weight distribution than `arbf` — more cells
looking "similarly close" to more neighbors under the same graph_construction/k settings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
colors = {list(aff.keys())[0]: '#9CA3AF', list(aff.keys())[1]: '#3B82F6'}

for name, M in aff.items():
    degree = np.asarray((M > 0).sum(axis=1)).ravel()
    axes[0].hist(degree, bins=60, alpha=0.6, label=name, color=colors[name], density=True)
    weights = M.data
    axes[1].hist(weights, bins=60, alpha=0.6, label=name, color=colors[name], density=True)

axes[0].set_title('Degree (n neighbors with weight > 0)', fontsize=10)
axes[0].set_xlabel('degree')
axes[1].set_title('Nonzero edge weight distribution', fontsize=10)
axes[1].set_xlabel('edge weight')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(fontsize=7, loc='upper right')
fig.tight_layout()
plt.show()

for name, M in aff.items():
    degree = np.asarray((M > 0).sum(axis=1)).ravel()
    print(f'{name}:')
    print(f'  degree  median={np.median(degree):.1f}  mean={degree.mean():.1f}  '
          f'p90={np.percentile(degree, 90):.1f}')
    print(f'  weight  median={np.median(M.data):.4f}  mean={M.data.mean():.4f}  '
          f'p90={np.percentile(M.data, 90):.4f}')

## Same-cell-type vs. cross-cell-type edge weights

The direct test of the smoothing hypothesis: for each graph, split its nonzero edges into
"same cell type" and "different cell type" and compare their weight distributions. If `ctx`
can't tell same-type from cross-type edges apart (distributions nearly identical) while
`arbf` clearly can (same-type edges systematically heavier), that's direct evidence the
spatial-context pooling is washing out the cell-type signal the affinity graph is supposed to
respect.

In [ ]:
def same_vs_cross_ct_weights(M, celltypes, max_edges=2_000_000, seed=0):
    """Sample nonzero edges (i, j, weight) and split by whether celltypes[i] == celltypes[j].
    Sampling keeps this cheap on a 28804x28804 sparse matrix with tens of millions of edges.
    """
    Mc = M.tocoo()
    n = Mc.nnz
    rng = np.random.default_rng(seed)
    if n > max_edges:
        idx = rng.choice(n, size=max_edges, replace=False)
        row, col, data = Mc.row[idx], Mc.col[idx], Mc.data[idx]
    else:
        row, col, data = Mc.row, Mc.col, Mc.data

    same = celltypes[row] == celltypes[col]
    return data[same], data[~same]

summary_rows = []
fig, axes = plt.subplots(1, len(aff), figsize=(5 * len(aff), 3.5), sharey=True)
if len(aff) == 1:
    axes = [axes]

for ax, (name, M) in zip(axes, aff.items()):
    same_w, cross_w = same_vs_cross_ct_weights(M, celltypes)
    ax.hist(same_w, bins=60, alpha=0.6, label=f'same cell type (n={len(same_w):,})',
            color='#3B82F6', density=True)
    ax.hist(cross_w, bins=60, alpha=0.6, label=f'cross cell type (n={len(cross_w):,})',
            color='#F59E0B', density=True)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('edge weight')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize=7)

    summary_rows.append({
        'graph': name,
        'mean_weight_same_ct': same_w.mean(),
        'mean_weight_cross_ct': cross_w.mean(),
        'ratio_same_over_cross': same_w.mean() / cross_w.mean() if cross_w.mean() > 0 else np.nan,
        'frac_edges_cross_ct': (len(cross_w) / (len(same_w) + len(cross_w))),
    })

fig.tight_layout()
plt.show()

summary = pd.DataFrame(summary_rows).set_index('graph')
print(summary.round(4).to_string())
print()
print('ratio_same_over_cross close to 1.0 means the graph barely distinguishes same-type '
      'from cross-type edges by weight (smoothing washed out cell-type signal). A ratio '
      'clearly > 1 (arbf, expected) means same-type edges are systematically heavier -- the '
      'graph still "knows" cell-type identity even though it was never told it explicitly.')

## Cross-check against the plan1 result: scProto metacell size distribution

Ties this back to `plan1_niche_recovery_eval.ipynb`'s finding directly: if the `ctx` graph
really is under-discriminating, the actual metacell sizes it produces (via
`cell_assignments.csv`, already saved from training) should show the predicted "few giant
attractor prototypes + many near-empty ones" pattern, in contrast to SEACells.

In [ ]:
from interpretable_ssl.evaluation.paper_figures import _resolve_run_dir

SCPROTO_KEYWORD = 'proto_umap_prtInit-wayp_aff-ctx_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp'
run_dirs = {
    'scProto': _resolve_run_dir(DS_ID, SCPROTO_KEYWORD),
    'SEACells': _resolve_run_dir(DS_ID, 'seacell'),
}

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, run_dir in run_dirs.items():
    sizes = pd.read_csv(os.path.join(run_dir, 'cell_assignments.csv')).groupby('metacell_id').size()
    print(f'{name}: n_metacells={len(sizes)}  min={sizes.min()}  median={sizes.median():.0f}  '
          f'mean={sizes.mean():.1f}  max={sizes.max()}  '
          f'frac_below_10_cells={(sizes < 10).mean():.1%}')
    ax.hist(np.log10(sizes.clip(lower=1)), bins=40, alpha=0.6, label=name,
            color=colors[list(aff.keys())[1]] if name == 'scProto' else colors[list(aff.keys())[0]],
            density=True)

ax.set_xlabel('log10(metacell size, n real member cells)')
ax.set_title('Metacell size distribution: scProto vs. SEACells', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Summary

- If the same-vs-cross-cell-type weight ratio is close to 1.0 for `ctx` but clearly above 1.0
  for `arbf`, that's direct evidence the spatial-context mean-pooling in `X_ctx`
  (`build_context`, radius=7.5, no cell-type filter) washes out cell-type-discriminative
  signal before the adaptive-RBF kernel ever sees it — consistent with the metacell
  size-skew seen both here and in `plan1_niche_recovery_eval.ipynb`.
- If that ratio is *not* close to 1.0 (arbf and ctx look similar), the collapse is more likely
  coming from elsewhere (e.g. the shared 300-prototype budget across 18 very unevenly-sized
  cell types, independent of the affinity graph) — worth checking `n_unused_protos` /
  `unused_proto_ratio` from this run's `metrics.json` next.
- Either way, this notebook only reads already-saved artifacts (graphs + `cell_assignments.csv`)
  — no retraining was needed to get this evidence. The next real experiment (if the affinity
  hypothesis holds up) is masking cross-cell-type edges out of the `ctx` graph and retraining,
  as discussed for Option 2 in the plan1 diagnostic follow-up.
